# Lecture: Your First CUDA Kernels (Threads, Blocks, Grids)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Draw the CUDA hierarchy (thread → block → grid) and compute a thread's global index with `i = blockIdx.x * blockDim.x + threadIdx.x` (§4–§5).
- Explain the tail guard (`if i < n`): with n=10 and block=4, threads 0–9 work and 10–11 sit out (§5).
- Name the three real kernel backends in the lab (Triton, numba-CUDA, C++-inline, §6), say what the CPU thread-simulator and reference prove (§7), and trace softmax's two passes (§8: max 2.0, sum 1.5174).

## The big idea

A CUDA **kernel** is a recipe written for one worker: 'add the two numbers at *my* position'. The GPU then hires thousands of workers (**threads**) and hands each one a numbered ticket. Workers are organized into groups called **blocks**, and blocks into a **grid** — think post-office counters (blocks) each serving a row of customers (threads). Each worker computes which data is *mine* from its ticket with one famous line, the **index idiom**: `i = blockIdx.x * blockDim.x + threadIdx.x` (my counter number × counter size + my seat number). Because counters come in fixed sizes, the last counter usually has spare seats: 10 elements with seats of 4 needs 3 counters (12 seats), so 2 extra workers check `if i < n` and sit out — that is the **tail guard**, and forgetting it means workers scribble past the end of your data. A **launch** (`<<<grid, block>>>`) hires grid×block workers to run the recipe, and **Triton**, **numba-CUDA**, and **C++-inline** are just three languages for mailing the same recipe to the GPU. Two safety nets: a **reference** kernel (the one-line torch version, e.g. `a + b`) is the answer key every custom kernel is checked against, and a **thread-simulator** (nested CPU loops over block/thread running the same recipe serially) proves the *logic* without a GPU. Softmax previews the next step up: a **two-pass** row recipe (max, then exp, then normalize — max 2.0, sum 1.5174 below) that Project 6 will optimize.

In [ ]:
import numpy as np  # bring in numpy for the hierarchy arithmetic
import matplotlib.pyplot as plt  # bring in pyplot for the hierarchy diagram
np.random.seed(8)  # fix the seed (geometry below is fixed; seed keeps the habit)
n, block = 4096, 256  # define lab-like launch: 4096 elements, 256 threads per block
grid = (n + block - 1) // block  # ceil division: grid = 16 blocks covering 4096 slots exactly
print('n =', n, 'block =', block, 'grid =', grid)  # print the launch geometry (16 blocks, no tail here)
print('threads hired:', grid * block, '(one per element)')  # print 4096: full coverage, nothing guarded
n2, b2 = 10, 4  # define the toy case from §5: 10 elements with seats of 4
g2 = (n2 + b2 - 1) // b2  # ceil division: grid = 3 blocks covering 12 slots
print('toy: n =', n2, 'block =', b2, 'grid =', g2, '(12 slots, 2 guarded off)')  # print the tail-guard preview
fig, ax = plt.subplots(figsize=(8, 3.5))  # create one wide figure with one axes
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 12)  # fix x-range for box placement
ax.set_ylim(0, 4)  # fix y-range for box placement
ax.text(6, 3.4, 'GRID (16 blocks)', ha='center', fontsize=10, fontweight='bold')  # grid banner on top
for bx in range(4):  # draw 4 representative blocks (of 16) as boxes
    ax.text(1.5 + bx * 3, 2.0, f'block {bx}\n256 threads', ha='center', va='center', fontsize=8,  # block label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # rounded blue box per block
ax.text(6, 0.7, '... 12 more blocks ...  ->  4096 threads, one element each (toy §5: 3 blocks, 12 slots, 2 guarded)', ha='center', fontsize=9)  # ellipsis line
ax.set_title('Anchor: grid of blocks of threads (thread -> block -> grid)')  # title the hierarchy
plt.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 hires the workers, §5 seats them, §6 ships the recipe, §7 checks the answers, §8 climbs to a two-pass recipe.

| Term | One-liner | Section |
|---|---|---|
| thread | One GPU worker running the kernel recipe on its own element | §4 |
| block (`blockDim` / `threadIdx`) | A group of threads (lab uses 256); `blockDim` = size, `threadIdx` = seat | §4 |
| grid (`blockIdx`) | All blocks of a launch; `blockIdx` = which block; grid = ceil(n / block) | §4 |
| global index idiom | `i = blockIdx.x * blockDim.x + threadIdx.x`: tickets → one data position | §5 |
| tail guard (`if i < n`) | Drops over-provisioned tail threads (ids 10, 11 for n=10, block=4) | §5 |
| kernel / launch | Kernel: the per-thread recipe. Launch (`<<<grid, block>>>`): hire grid×block workers | §6 |
| Triton / numba-CUDA / C++-inline | Three ways to ship a recipe to the GPU (DSL / decorator / raw CUDA via torch) | §6 |
| thread-simulator | CPU nested loops running the same recipe serially — proves logic, not speed | §7 |
| reference kernel | The one-line torch oracle (e.g. `a + b`) every custom kernel is checked against | §7 |
| softmax two-pass | Row max, then exp, then normalize (max 2.0, sum 1.5174 on the worked row) | §8 |

### §4 Thread, block, and grid — hiring the workers

**In one sentence: a thread is one GPU worker, threads are grouped into blocks (the lab uses 256 per block), blocks form the grid, and a launch hires grid × block workers where grid = ceil(n / block).**

Post-office picture: each customer (thread) sits at a seat (`threadIdx`) at one counter (**block**, size `blockDim`), and all counters together are the **grid** (`blockIdx` says which counter). For the lab's 4096-element vectors with 256-thread blocks: grid = 16, hiring exactly 4096 workers — full coverage, no tail. For the toy 10-element case with seats of 4: grid = 3, hiring 12 workers for 10 elements (§5 seats them). Blocks matter because the GPU schedules whole blocks onto multiprocessors — block size is the occupancy knob Project 6 tunes. Watch it: the hierarchy diagram with both geometries printed.

In [ ]:
import numpy as np  # reuse numpy for the grid arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the hierarchy diagram
np.random.seed(8)  # fix the seed (geometry below is fixed; seed keeps the habit)
n, block = 4096, 256  # lab geometry: 4096 elements, 256 threads per block
grid = (n + block - 1) // block  # ceil division: 16 blocks, exact coverage
print('lab: n =', n, 'block =', block, 'grid =', grid, '-> threads:', grid * block)  # print 16 blocks, 4096 threads
n2, b2 = 10, 4  # toy geometry for §5: 10 elements, seats of 4
g2 = (n2 + b2 - 1) // b2  # ceil division: 3 blocks, 12 slots
print('toy: n =', n2, 'block =', b2, 'grid =', g2, '-> threads:', g2 * b2)  # print 3 blocks, 12 threads
print('rule: grid = ceil(n / block); launch hires grid x block threads')  # print the hiring rule
fig, ax = plt.subplots(figsize=(8, 4))  # create one figure with one axes
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 12)  # fix x-range for box placement
ax.set_ylim(0, 5)  # fix y-range for two rows
ax.text(6, 4.4, 'GRID: 3 blocks (toy n=10, block=4)', ha='center', fontsize=10, fontweight='bold')  # grid banner
for bx in range(3):  # draw the 3 toy blocks
    ax.text(2 + bx * 4, 3.0, f'block {bx}\n4 threads', ha='center', va='center', fontsize=9,  # block label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # rounded blue box per block
    for tx in range(4):  # draw 4 thread seats inside each block
        ax.text(0.9 + bx * 4 + tx * 0.75, 1.6, f't{tx}', ha='center', va='center', fontsize=8,  # thread seat label
                bbox=dict(boxstyle='circle', facecolor='wheat'))  # wheat circle per thread
ax.text(6, 0.5, 'thread -> block -> grid: 12 tickets for 10 elements (§5 guards 2)', ha='center', fontsize=9)  # moral line
ax.set_title('Hierarchy: threads live in blocks, blocks live in the grid')  # title the organization
plt.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

### §5 The index idiom and the tail guard — seating the workers

**In one sentence: each thread computes its data position with `i = blockIdx.x * blockDim.x + threadIdx.x`, and the tail guard `if i < n` drops the over-provisioned threads of the last block (for n=10, block=4: threads 0–9 work, 10–11 sit out).**

Every kernel uses the idiom — block number × block size + seat number — so tickets never collide and never skip. Ceil-division always over-provisions unless `n` divides evenly, so the guard is not optional: without it, tail threads read and write past the arrays (garbage or crashes). In vectorized Triton code the same guard appears as a mask (`offs < n`) instead of an `if`. Watch it: the green/red element map — 12 slots, 10 green workers, 2 red guarded-off.

In [ ]:
import numpy as np  # bring in numpy for the index arithmetic
import matplotlib.pyplot as plt  # bring in pyplot for the thread-mapping plot
np.random.seed(8)  # fix the seed (mapping below is fixed; seed keeps the habit)
n = 10  # define data size: 10 elements to process (indices 0..9)
block = 4  # define block size: 4 threads per block (tiny stand-in for real 256)
grid = (n + block - 1) // block  # compute grid size with ceil division: exactly like <<<grid, block>>>
print('n =', n, 'block =', block, 'grid =', grid)  # print the launch geometry (3 blocks cover 12 slots)
global_ids = []  # create list collecting every thread's global index across all blocks
block_ids = []  # create list recording each slot's block number for labels
thread_ids = []  # create list recording each slot's thread number for labels
for bx in range(grid):  # loop over blocks (on GPU these run at the same time)
    for tx in range(block):  # loop over threads inside one block
        i = bx * block + tx  # compute the GLOBAL index: the canonical CUDA idiom
        global_ids.append(i)  # record this thread's global index
        block_ids.append(bx)  # record the block number for the x label
        thread_ids.append(tx)  # record the thread number for the x label
global_ids = np.array(global_ids)  # convert indices to an array for masking
valid = global_ids < n  # build the tail-guard mask: threads with i < n do real work
print('valid thread ids:', global_ids[valid].tolist())  # print working threads 0..9 (one per element)
print('guarded-off ids:', global_ids[~valid].tolist())  # print idle threads [10, 11] (dropped by if i < n)
colors = ['green' if v else 'red' for v in valid]  # color workers green and guarded tail threads red
ticklabels = ['b' + str(b) + '/t' + str(t) for b, t in zip(block_ids, thread_ids)]  # label each slot block/thread
fig, ax = plt.subplots(figsize=(7, 3.5))  # create one wide figure with one axes
ax.bar(range(len(global_ids)), np.ones(len(global_ids)), color=colors)  # draw one unit bar per thread slot
for k, i in enumerate(global_ids):  # loop over thread slots to annotate each bar
    ax.text(k, 0.5, str(int(i)), ha='center', va='center')  # write the global index inside its bar
ax.set_xticks(range(len(global_ids)))  # place one tick per thread slot
ax.set_xticklabels(ticklabels, fontsize=8)  # name each tick after its block/thread slot
ax.set_xlabel('thread slot (block / thread)')  # label x as thread slots grouped by block
ax.set_ylabel('one element of work')  # label y as a unit of work per thread
ax.set_title('12 threads cover 10 elements: green work, red guarded off (if i < n)')  # plain-English title
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

### §6 Kernel, launch, and backends — shipping the recipe

**In one sentence: a kernel is the per-thread recipe (`out[i] = a[i] + b[i]` with `i` from the idiom), a launch (`<<<grid, block>>>`) hires the workers to run it, and Triton (Python-like DSL with `program_id` + masked loads), numba-CUDA (`@cuda.jit` with `cuda.grid(1)`), and C++-inline (raw CUDA compiled via torch) are three backends that ship the identical recipe — all need a GPU plus toolchain to actually launch.**

Same geometry everywhere: for n = 4096 and block = 256, every backend computes grid = 16 — the table below is one row per backend with its spelling of the idiom and guard. Triton's mask (`offs < n`) is the vectorized tail guard; numba's `cuda.grid(1)` is the same `bx*block+tx` in new spelling; the C++ source in the lab is 12 lines (kernel + launcher) echoed then compiled only when nvcc + CUDA exist. Watch it: the launch-config diagram — one recipe, three envelopes, one geometry (16 blocks cover 4096).

In [ ]:
import numpy as np  # reuse numpy for the launch-geometry math
import matplotlib.pyplot as plt  # reuse pyplot for the launch-config diagram
np.random.seed(9)  # fix the seed (geometry below is fixed; seed keeps the habit)
n, block = 4096, 256  # define the lab vector size and threads-per-block
grid = (n + block - 1) // block  # ceil division: 16 blocks (same number every backend computes)
print('n =', n, 'block =', block, 'grid =', grid)  # print 16: the shared launch geometry
print('triton: pid * BLOCK + arange(BLOCK), mask = offs < n (vector guard)')  # print Triton's spelling
print('numba: i = cuda.grid(1); if i < n (same idiom, new spelling)')  # print numba's spelling
print('cpp-inline: i = blockIdx.x * blockDim.x + threadIdx.x; if (i < n)')  # print raw CUDA spelling
print('all three need a GPU + toolchain to LAUNCH; all share grid = 16')  # print the launch rule
fig, ax = plt.subplots(figsize=(9, 3.2))  # make a wide flat figure for the launch diagram
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix x-range for box placement
ax.set_ylim(0, 3)  # fix y-range for box placement
ax.text(2.0, 1.5, 'recipe\nout[i]=a[i]+b[i]\n(i from idiom)', ha='center', va='center', fontsize=9,  # the shared recipe box
        bbox=dict(boxstyle='round', facecolor='lightgreen'))  # green = one shared recipe
for j, name in enumerate(['triton\nmask guard', 'numba\nif guard', 'cpp-inline\nif guard']):  # three backend envelopes
    ax.text(6.0 + j * 2.6, 1.5, name, ha='center', va='center', fontsize=8,  # backend envelope label
            bbox=dict(boxstyle='round', facecolor='wheat'))  # wheat = envelope around the recipe
    ax.annotate('', xy=(4.9 + j * 2.6, 1.5), xytext=(3.4, 1.5), arrowprops=dict(arrowstyle='->'))  # arrow recipe -> envelope
ax.text(7, 2.6, 'ONE recipe, THREE envelopes; every launch: grid=16, block=256', ha='center', fontsize=9, fontweight='bold')  # banner line
ax.set_title('Launch config: <<<16, 256>>> hires 4096 workers in every backend')  # title the shared geometry
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram

### §7 Simulator and reference — checking answers without a GPU

**In one sentence: the reference is the one-line torch oracle (`a + b`) that defines the right answer, and the thread-simulator is nested CPU loops over (block, thread) running the same guarded recipe serially — it proves the kernel *logic* is correct on any machine, while saying nothing about GPU *speed*.**

Below: 8 elements, block 4 → grid 2, every thread computes `i = bx*block + tx`, guards `i < 8`, and writes `out[i] = a[i] + b[i]` — then the max difference vs the reference prints 0.0. The lab runs the same trick for relu, softmax, and reduction (reduction adds a second act: each block writes one partial sum, then a tiny final sum — the preview of Project 6's tree). Graceful rule: GPU launches run only when CUDA is present; otherwise every GPU backend row records SKIPPED and the simulator carries the verification. Watch it: the emulation numbers plus the match certificate.

In [ ]:
import numpy as np  # reuse numpy for the reference + simulator math
import matplotlib.pyplot as plt  # reuse pyplot for the emulation schematic
np.random.seed(42)  # fix the seed so the test vectors match the lab's seed habit
a = np.array([0.5, -1.5, 2.0, 0.0, 1.0, -0.5, 3.0, -2.0])  # define test vector a (8 hand-picked values incl. negatives)
b = np.array([1.0, 1.0, -1.0, 2.0, 0.5, 0.5, -3.0, 2.0])  # define test vector b (same length)
ref = a + b  # REFERENCE: the one-line oracle answer key (stands in for torch a+b)
n, block = 8, 4  # simulator geometry: 8 elements, 4 threads per block
grid = (n + block - 1) // block  # ceil division: 2 blocks, exact coverage here
out = np.empty_like(a)  # output buffer standing in for GPU global memory
for bx in range(grid):  # loop over blocks serially (a GPU would run them concurrently)
    for tx in range(block):  # loop over threads in the block
        i = bx * block + tx  # GLOBAL index: the canonical idiom, CPU spelling
        bod = i < n  # tail guard as an explicit boolean (same job as if i < n)
        out[i] = a[i] + b[i] if bod else out[i]  # guarded body: workers past the end sit out
print('reference:', np.round(ref, 2).tolist())  # print the oracle answers
print('simulator:', np.round(out, 2).tolist())  # print the emulated answers (must match)
print('max diff:', float(np.abs(out - ref).max()), '(0.0 = logic correct)')  # print 0.0: the match certificate
print('speed note: serial loops prove LOGIC; parallelism (thousands at once) is the speed')  # print the honesty note
fig, ax = plt.subplots(figsize=(7, 2.6))  # make a flat figure for the emulation schematic
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 10)  # fix x-range for box placement
ax.set_ylim(0, 3)  # fix y-range for box placement
ax.text(1.8, 1.5, 'reference\na + b\n(answer key)', ha='center', va='center', fontsize=9,  # reference box
        bbox=dict(boxstyle='round', facecolor='lightgreen'))  # green = trusted oracle
ax.text(5.0, 1.5, 'simulator\nbx/tx loops\n+ guard', ha='center', va='center', fontsize=9,  # simulator box
        bbox=dict(boxstyle='round', facecolor='wheat'))  # wheat = serial emulation
ax.text(8.2, 1.5, 'compare\nmax diff 0.0', ha='center', va='center', fontsize=9,  # compare box
        bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = verdict
ax.annotate('', xy=(3.4, 1.5), xytext=(2.9, 1.5), arrowprops=dict(arrowstyle='->'))  # arrow reference -> simulator
ax.annotate('', xy=(6.9, 1.5), xytext=(6.1, 1.5), arrowprops=dict(arrowstyle='->'))  # arrow simulator -> compare
ax.set_title('Check without a GPU: emulate guarded recipe, compare to oracle')  # title the verification habit
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram

### §8 Softmax two-pass — max, then exp-and-sum

**In one sentence: row-softmax is a two-pass recipe — pass 1 finds the row max and exponentiates the stabilized values, pass 2 sums them and divides — because exponentiation overflows without the max subtraction and normalization needs the sum first.**

Worked row `[2.0, 1.0, 0.1]`: pass 1a takes the max, 2.0 (subtracting it changes nothing mathematically — it cancels in the division — but prevents `exp(1000) = inf`). Pass 1b exponentiates the shifted row `[0, -1, -1.9]` → `[1.0, 0.3679, 0.1496]`. Pass 2 sums (1.5174) and divides → probabilities `[0.659, 0.242, 0.099]`, which sum to 1. A real kernel gives one block per row so rows run in parallel; Triton's version fuses both passes in on-chip memory (no second global read), which is the Project 6 preview. Watch it: the pass-by-pass numbers plus the before/after bars.

In [ ]:
import numpy as np  # reuse numpy for the two-pass softmax math
import matplotlib.pyplot as plt  # reuse pyplot for the before/after bars
np.random.seed(5)  # fix the seed (row below is fixed; seed keeps the habit)
row = np.array([2.0, 1.0, 0.1])  # define one worked logit row (true class: index 0)
m = row.max()  # PASS 1a: row max = 2.0 (stability shift; cancels in the division)
shifted = row - m  # stabilize: [0, -1, -1.9] (same distribution, no overflow risk)
e = np.exp(shifted)  # PASS 1b: stabilized exponentials [1.0, 0.3679, 0.1496]
s = e.sum()  # PASS 2a: row sum = 1.5174 (normalizer; needs the full row first)
p = e / s  # PASS 2b: normalize to probabilities [0.659, 0.242, 0.099]
print('row max:', m)  # print 2.0: the stability shift
print('exp(shifted):', np.round(e, 4))  # print [1.0, 0.3679, 0.1496]: pass-1 output
print('sum:', round(float(s), 4))  # print 1.5174: the pass-2 normalizer
print('probs:', np.round(p, 3), 'sum =', round(float(p.sum()), 6))  # print [0.659 0.242 0.099], contract = 1.0
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))  # make two side-by-side panels
ax[0].bar(['c0', 'c1', 'c2'], row, color='steelblue')  # left: raw logits (can exceed 1, any values)
ax[0].set_title('before: logits')  # left title
ax[0].set_ylabel('score')  # left label
ax[1].bar(['c0', 'c1', 'c2'], p, color='darkorange')  # right: probabilities (0..1, sum 1)
ax[1].set_title('after: probabilities (sum = 1)')  # right title
ax[1].set_ylabel('probability')  # right label
plt.suptitle('Softmax two-pass: max 2.0, then exp, then divide by 1.5174')  # overall takeaway with the numbers
plt.tight_layout()  # snug the layout
plt.show()  # render the figure

## How it all fits together — the system map §9

One kernel delivery is a pipeline with every § term playing a position. **Hire** (§4): pick block (256) and compute grid = ceil(n/block). **Seat** (§5): each thread computes `i = bx*block + tx` and the tail guard drops `i >= n`. **Ship** (§6): the same recipe goes out via Triton, numba, or C++-inline — one geometry, three envelopes. **Check** (§7): the CPU simulator replays the guarded recipe serially and the reference oracle certifies max-diff 0.0. **Level up** (§8): multi-pass recipes (softmax: max 2.0, sum 1.5174) reuse the same seating with per-row blocks. The map below pins each term to its station; the numbers trace one block's tickets.

In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
fig, ax = plt.subplots(figsize=(11, 3.8))  # wide figure for the delivery pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('hire\ngrid/block\n§4', 1.2), ('seat\ni + guard\n§5', 3.8), ('ship\n3 backends\n§6', 6.4), ('check\nsim vs ref\n§7', 9.2), ('level up\nsoftmax\n§8', 12.0)]  # pipeline stations in order
for name, xc in stages:  # draw each station box
    ax.text(xc, 2.5, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = delivery pipeline
for xa, xb in [(1.9, 3.0), (4.6, 5.6), (7.3, 8.4), (10.1, 11.1)]:  # gaps between stations
    ax.annotate('', xy=(xb, 2.5), xytext=(xa, 2.5), arrowprops=dict(arrowstyle='->'))  # forward arrows left to right
ax.text(7, 3.6, 'ONE KERNEL DELIVERY: hire -> seat -> ship -> check -> level up', ha='center', fontsize=9, fontweight='bold')  # phase banner
ax.text(7, 0.5, 'same i = bx*block+tx + guard everywhere; oracle certifies 0.0 diff', ha='center', fontsize=9, fontweight='bold', color='darkred')  # red = the two habits
ax.set_title('System map §9: launch config → thread → element → guarded compute → compare')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map

In [ ]:
import numpy as np  # reuse numpy for the one-block index trace
np.random.seed(9)  # fix the seed (indices below are fixed; habit kept)
n, block, bx = 10, 4, 1  # trace setup: the toy launch, block number 1
tickets = [bx * block + tx for tx in range(block)]  # block 1 tickets: 1*4 + [0,1,2,3] = [4, 5, 6, 7]
guard = [i < n for i in tickets]  # guard verdicts: all True (block 1 is fully covered)
print('block 1 tickets:', tickets)  # print [4, 5, 6, 7]: every seat maps inside the data
print('guard:', guard, '(all work)')  # print all-True: no tail in the middle block
bx2 = 2  # now the tail block of the same launch
tickets2 = [bx2 * block + tx for tx in range(block)]  # block 2 tickets: [8, 9, 10, 11]
guard2 = [i < n for i in tickets2]  # guard verdicts: 8,9 work; 10,11 sit out
print('block 2 tickets:', tickets2)  # print [8, 9, 10, 11]: two past the end
print('guard:', guard2, '(10, 11 guarded off)')  # print the tail-guard action

## Bridge to the lab

Open `notebook.ipynb` now. It writes the same five ops as raw kernels and checks each against torch:

1. **Cell 1 (goal markdown)** — the hierarchy diagram (`thread → block → grid`) and the index idiom. You can already recite both (§4–§5).
2. **Cell 2 (environment probe)** — seeds plus backend flags: `HAS_CUDA`, `HAS_TRITON`, `HAS_NUMBA`, `HAS_NVCC`, `HAS_SMI` (§6). GPU launches run only when the flags allow; everything else degrades to the simulator with SKIPPED rows.
3. **Cell 3 (torch references)** — one-line oracles (`a+b`, `relu`, `sigmoid`, stable `softmax`, `sum`) verified against NumPy (§7). These are the answer keys.
4. **Cell 4 (CPU thread-simulator)** — `cuda_vecadd_cpu` / `cuda_relu_cpu` / `cuda_softmax_cpu` / `cuda_reduce_cpu`: the exact `i = bx*block + tx` + `if i < n` pattern (§5, §7), run as serial Python loops with the 0.0-diff certificate.
5. **Cell 5 (Triton kernels)** — real `@triton.jit` code (`vecadd`, `sigmoid`, row-`softmax` with `tl.arange` + mask guard = the vectorized tail guard, §6, §8). Defined always, launched only with CUDA (grid 16 for n=4096).
6. **Cell 6 (numba + C++-inline)** — `@cuda.jit` kernels with `cuda.grid(1)` (same idiom, new spelling, §6) and the 12-line raw CUDA source echoed then compiled via `load_inline` when nvcc + CUDA exist.
7. **Cell 7 (benchmark + chart)** — torch vs NumPy vs thread-simulator at n=200k (serial ≈1000x slower — parallelism is the point, §7), the block-size sweep framed honestly as an analogy (§4's occupancy knob), `results/kernels.csv`, and the log-scale bar chart.
8. **Cell 8 (README prompts)** — the one-liner: a kernel is a per-thread recipe (§6); speed is thousands of recipes at once (§4).

## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **With n=10 and block=4, which threads do real work and which sit out — and why?**

*Answer: grid = ceil(10/4) = 3 blocks = 12 threads (§4). Threads 0–9 process one element each via `i = bx*4 + tx`; threads 10–11 (block 2's tail seats, §9 trace) are dropped by the* `if i < n` *guard (§5) — the red bars in the §5 map.*

2. **What breaks if you delete the `if i < n` guard?**

*Answer: the over-provisioned tail threads read and write past the end of the arrays (§5) — out-of-bounds memory access, garbage results or crashes. Triton's `mask = offs < n` (§6) is the same guard in vector form.*

3. **The CPU simulator matches the reference (max diff 0.0) — so why isn't it as good as a GPU kernel, and what does softmax's second pass need?**

*Answer: the simulator runs the identical guarded recipe serially (§7), so logic matches but speed comes from thousands of threads at once, not a different formula. Softmax's pass 2 (§8) needs pass 1's sum (1.5174) first — normalization cannot start until the whole row's exponentials exist.*